# Q5. Maximum set cover, minimum edge cover and maximum independent set

Lecture: *Week 6 - Integer Linear Programming - Further Problems*, last slide (*Tutorial*), task 5: look up other
covering problems and code them yourself. This notebook answers it with this week's two examples, so you can
compare each new model with the one from the lecture. **Try the task before you open it.**

- **(a) Maximum set cover**, on the fire stations (demos 1–3): with at most $k = 2$ stations, cover as many
  cities as possible.
- **(b) Minimum edge cover**, on the security-camera graph (demo 4): choose as few corridors (edges) as
  possible so that every corner (vertex) touches a chosen corridor.
- **(c) Maximum independent set**, on the same graph: choose as many corners as possible with no two of them
  joined by a corridor.

Task 2 (the fire stations written with an incidence matrix) is answered by demo 2,
`2 Lecture Demos/2 Costed fire stations - incidence matrix.ipynb`.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes
about a minute), then *Run → Run All Cells*. In the browser HiGHS prints its log after each solve; the results
are not affected.

**Expected results.** (a) **4 cities** of 5 with two stations; (b) **4 corridors**; (c) **4 corners**. Each
problem has several optimal answers, so your selection may differ from the one printed here.

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo and create the solver once; each part below builds its own model.

In [ ]:
from pyomo.environ import *

solver = SolverFactory('appsi_highs')

## (a) Maximum set cover: fire stations, at most two of them

Same data as demo 3: station $t$ covers the cities in $s_t$ (cities numbered 0–4, as on the slides). Now there
is no cost; instead at most $k$ stations may be opened, and we count the cities covered.

$x_t = 1$ if station $t$ is opened, $y_e = 1$ if city $e$ counts as covered:

maximise $\sum_{e \in U} y_e$ subject to $y_e \le \sum_{t:\, e \in s_t} x_t$ for every city $e$,
$\sum_{t \in S} x_t \le k$, all variables binary.

$y_e$ can only be 1 if at least one opened station covers city $e$.

In [ ]:
stations = ['a', 'b', 'c', 'd']
cities = [0, 1, 2, 3, 4]
covers = {'a': [0, 2], 'b': [1, 2, 3], 'c': [1, 4], 'd': [4]}   # s_t from the slides
k = 2                                                           # at most two stations

m1 = ConcreteModel()
m1.x = Var(stations, within=Binary)   # x[t] = 1 if station t is opened
m1.y = Var(cities, within=Binary)     # y[e] = 1 if city e is covered

# Count the covered cities
m1.obj = Objective(expr=sum(m1.y[e] for e in cities), sense=maximize)

# A city counts only if an opened station covers it (one row per city)
def covered_rule(m, e):
    return m.y[e] <= sum(m.x[t] for t in stations if e in covers[t])
m1.covered = Constraint(cities, rule=covered_rule)

# Open at most k stations
m1.budget = Constraint(expr=sum(m1.x[t] for t in stations) <= k)

solver.solve(m1)
print('Cities covered:', value(m1.obj))
print('Stations opened:', [t for t in stations if value(m1.x[t]) > 0.5])
print('Cities covered: ', [e for e in cities if value(m1.y[e]) > 0.5])

**Answer.** 4 of the 5 cities. No two stations cover all five: city 0 needs $a$ and city 3 needs $b$, and
$a$ and $b$ together miss city 4. Several pairs reach 4 ($a$ + $b$, $a$ + $c$, $b$ + $c$, $b$ + $d$), so the
solver may print any of them; HiGHS prints $a$ + $c$.

## (b) Minimum edge cover: security cameras graph

The graph of demo 4: corners 1–8 and ten corridors. An *edge cover* is a set of edges that touches every vertex
(the hidden slides 10–11 of the lecture `.pptx` show examples). $z_{uv} = 1$ if edge $\{u, v\}$ is chosen:

minimise $\sum_{\{u,v\} \in E} z_{uv}$ subject to $\sum_{\{u,v\} \ni w} z_{uv} \ge 1$ for every vertex $w$,
$z_{uv} \in \{0, 1\}$.

Compare with vertex cover: there the variables were vertices and each constraint an edge; here the roles are
swapped.

In [ ]:
vertices = range(1, 9)
edges = [(1, 2), (1, 8), (2, 3), (2, 5), (3, 4),
         (4, 5), (5, 6), (5, 8), (6, 7), (7, 8)]   # the ten constraints on the slides

m2 = ConcreteModel()
m2.z = Var(edges, within=Binary)   # z[u, v] = 1 if corridor u-v is chosen

# Use as few corridors as possible
m2.obj = Objective(expr=sum(m2.z[e] for e in edges), sense=minimize)

# Every corner touches at least one chosen corridor (one row per vertex)
def touch_rule(m, w):
    return sum(m.z[e] for e in edges if w in e) >= 1
m2.touch = Constraint(vertices, rule=touch_rule)

solver.solve(m2)
print('Corridors needed:', value(m2.obj))
print('Chosen corridors:', [e for e in edges if value(m2.z[e]) > 0.5])

**Answer.** 4 corridors. Each corridor touches two of the 8 corners, so at least $8 / 2 = 4$ are needed, and
4 is possible: for example 1–8, 2–3, 4–5, 6–7 (what HiGHS prints) or 1–2, 3–4, 5–6, 7–8. Four corridors that touch all eight corners share no corner, so
they are also a *perfect matching* (slide 21).

## (c) Maximum independent set: same graph

An *independent set* is a set of vertices with no edge between any two of them. $x_u = 1$ if vertex $u$ is
chosen:

maximise $\sum_{u \in V} x_u$ subject to $x_u + x_v \le 1$ for every edge $\{u, v\}$, $x_u \in \{0, 1\}$.

It is demo 4 with the objective turned round and $\ge 1$ changed to $\le 1$.

In [ ]:
m3 = ConcreteModel()
m3.x = Var(vertices, within=Binary)   # x[u] = 1 if corner u is chosen

# Choose as many corners as possible
m3.obj = Objective(expr=sum(m3.x[u] for u in vertices), sense=maximize)

# The two ends of a corridor cannot both be chosen (one row per edge)
def apart_rule(m, u, v):
    return m.x[u] + m.x[v] <= 1
m3.apart = Constraint(edges, rule=apart_rule)

solver.solve(m3)
print('Largest independent set:', value(m3.obj))
print('Chosen corners:', [u for u in vertices if value(m3.x[u]) > 0.5])

**Answer.** 4 corners, for example 1, 3, 5, 7 or 2, 4, 6, 8.

## What do you notice?

Demo 4 needed 4 cameras, and the largest independent set here also has 4 corners, out of 8. Is that a
coincidence?

<details><summary>Answer</summary>

No. A set of vertices is independent exactly when the *other* vertices form a vertex cover: an edge with no
end in the cover would have both ends in the independent set. So the largest independent set and the smallest
vertex cover always add up to the number of vertices, here $4 + 4 = 8$. The two cover solutions on the slides
(2, 4, 6, 8 and 1, 3, 5, 7) are each the other's independent set.
</details>